# Experiment 2: Targeted Reranking

Inspect retrieval over the 15,585 eligible questions and reranking of 500 seeded opportunities plus 200 seeded controls. Each selected question retains the same thirty candidate passages.

**Live execution is the default.** `MODE="live"` runs this notebook’s model calls with the configured OpenAI-compatible client. Set `OPENROUTER_API_KEY` before starting the kernel. Prepared document embeddings and frozen experimental inputs remain fixed; passage embeddings are not rebuilt.

**Reference input scope:** the frozen larger corpus contains **152,576 passages** associated with **15,585 fully covered question IDs** from MS MARCO v2.1 validation; this is a subset, not the full validation split. Experiment 1 uses a separate **10,210-passage corpus**, 31 seeded baseline questions and a separate IMEI diagnostic. Experiment 2 samples **500 opportunities + 200 controls**. Experiment 3 contains **79 sampled pairs, including IMEI**. Experiment 4 times **8 question pairs / 16 answers**. IMEI is excluded from the 31-question baseline averages but is already included in the frozen Experiment 2 and Experiment 3 samples; displaying it again adds no observation.

**Workflow:** Experiment 1 (Baseline) --> Diagnosis --> **this Experiment 2 (Targeted Reranking)** --> Experiment 3 (Answer Quality) --> Experiment 4 (Latency Tradeoff)


**Recorded reference run:** 10 November 2026. Saved displays belong to that execution. A new Live run computes its own results; use its current tables rather than treating reference findings as predictions.

## 1. Set up the experiment

**What:** compare embedding top five with reranked top five from the same 30 candidates.

**Why:** test evidence selection while keeping the generator’s passage budget fixed.

Configuration and frozen inputs live beside this notebook. Live model calls read credentials only from the environment; results are written under the current run identity.

In [1]:
STUDY = 'targeted'
from pathlib import Path
import os, sys
from IPython.display import display, Markdown, HTML
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "configs/tutorial_inputs.json").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from shared_scoring.live_tutorial import TutorialRun
from shared_scoring.provider import embedding_payload, rerank_payload
from shared_scoring.tutorial_views import TutorialViews
from shared_scoring.notebook_display import display_table, details_html
from shared_scoring.io import read_json
import inspect, json
import pandas as pd

MODE = os.environ.get("NOTEBOOK_MODE", "live").lower()  # live | replay | plan
RUN_ID = os.environ.get("TUTORIAL_RUN_ID") or None
RESUME = os.environ.get("TUTORIAL_RESUME") == "1"  # explicit same-run resume only

tutorial = TutorialRun(PROJECT, STUDY, mode=MODE, run_id=RUN_ID, resume=RESUME)
tutorial.preflight(create_client=False)  # assets, credentials and offline tests; no model requests
client = None
if MODE == "live":
    from openai import OpenAI
    import httpx
    # Standard OpenAI-style client; the transport records this run and enforces caps.
    client = OpenAI(
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url=tutorial.retrieval["api_base"],
        max_retries=0,
        timeout=tutorial.retrieval["timeout_seconds"],
        http_client=httpx.Client(transport=tutorial.transport()),
    )
    tutorial.bind_client(client)
# Selecting Live authorizes this notebook's declared workflow; no second paid-call switch.

view = TutorialViews(tutorial)
view.show("configuration")


Setting,Value
Embedding model,qwen/qwen3-embedding-4b
Embedding dimensions,2560
Reranker model,qwen/qwen3-reranker-8b
Generator model,qwen/qwen3-30b-a3b-instruct-2507
Candidate / context budget,30 / 5 passages
Eligible questions in this experiment,700
Generation limit,400 tokens


## 2. Define the opportunity group and controls

**Recorded reference input selection:** the counts below describe how the frozen cohort was prepared, not a new diagnosis result.

**Opportunity:** embedding Recall@30 > Recall@5. The frozen cohort preparation identified **1,085** such questions; **500** were sampled before this Live reranking run. The new diagnosis is reported separately and does not resample these inputs.

**Controls:** **200** labeled no-gap questions sampled with seed 42 where Recall@30 = Recall@5. They test whether reranking loses evidence embedding already selected. All 200 selected controls in this run have Recall@5 = Recall@30 = 1.

Question selection preceded reranking and answer evaluation. Preserve the frozen IDs and recorded selection policy.

**Sampling-share clarification:** the “Opportunity share” table below divides the selected **500** by the upstream denominators; its **3.21% / 4.88%** values are sampling shares. In the frozen cohort’s preparation input, opportunity prevalence is **1,085 / 15,585 = 6.96%**, or **1,085 / 10,242 = 10.59%** among annotated questions.


In [2]:
view.show('questions')


subgroup
control        200
opportunity    500
Name: questions, dtype: int64

query_id,question,subgroup,diagnostic
1090267,botz etf price,control,False
query_id,question,subgroup,diagnostic
147500,difference between guitar and bass guitar,control,False
68095,can hives be a sign of pregnancy,control,False
106620,cost of repainting cabinets,control,False
281248,how many days does the landlord give you to pay your security deposit,control,False
1090206,synergistic synonym,control,False
207859,how are the months split into seasons,control,False
1090192,bowen law,control,False
49174,batter definition,control,False


## 3. Reuse the frozen inputs

**Frozen** means each question keeps its exact 30 retrieved candidate passages: IDs, text and original embedding order. Reranking changes their order; it does not retrieve new passages.

The frozen corpus contains 152,576 already embedded unique passages. The selected 500 opportunities and 200 controls retain 21,000 frozen candidate rows. Live execution reranks these frozen candidates without retrieving new passages.

Reference policy: include the pinned dataset’s original and well-formed answers, deduplicated without changing their text. This accepts alternative annotated wordings; the same fixed references score both paths. Do not compare this F1 directly with baseline F1: the question population, corpus and possibly reference policy differ.


In [3]:
view.show('input_contract')


{'questions': 700, 'candidate_rows': 21000, 'candidates_per_question': 30}

## 4. What runs at this stage

Live execution calls the Qwen reranker for each of the configured 500 opportunities and 200 controls, using each question’s frozen thirty candidates. Passage embedding stays disabled; the index uses only the checkpointed vectors.

Generation and answer scoring remain separate from the targeted retrieval experiment and are introduced in section 8.

In [4]:
def rerank_api(question, candidates):
    # OpenRouter rerank is exposed through the OpenAI client's generic POST API.
    return client.post("/rerank", cast_to=dict[str, object], body=rerank_payload(
        tutorial.retrieval["reranker_model"], question, candidates))

reranking_run = tutorial.rerank(rerank_api)
if reranking_run is not None:
    display(reranking_run)
retrieval_metrics_run = tutorial.diagnose()
if retrieval_metrics_run is not None:
    display(retrieval_metrics_run)


rerank: saved for CURRENT LIVE RUN 20261111T032855Z-3fa453244592


{'questions': 700,
 'candidate_rows': 21000,
 'run_id': '20261111T032855Z-3fa453244592'}

diagnose: saved for CURRENT LIVE RUN 20261111T032855Z-3fa453244592


{'annotated_questions': 700,
 'opportunities': 500,
 'run_id': '20261111T032855Z-3fa453244592'}

## 5. Compare evidence coverage

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

**Annotated-support Recall@5:** unique annotated support IDs in the selected five divided by all annotated support IDs. Missing support remains in the denominator. **First support rank:** position of the earliest annotated support passage.

Gain / unchanged / loss describes the paired change in Recall@5; ordering can improve without changing coverage.

In this **recorded reference run’s 500 opportunities**, mean recall changes **0.0638 → 0.6080**: **293 gains, 203 unchanged, 4 losses**. In **200 controls**, it changes **1.0000 → 0.9500**, with **10 losses**. Keep groups separate; their chosen sampling proportions do not represent population-wide benefit.

In [5]:
view.show('retrieval_summary')


diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta
0.0000,1.0100,1.0000,1.0000,1.8200,1.0000,0.7296,0.7971,0.9500,1.0000,1.9500,0.9500,0.7194,0.7773,0.0000,-0.0500
0.0020,1.1940,0.0638,0.9968,8.2720,0.1280,0.0834,0.0579,0.6080,0.9968,5.2640,0.6420,0.3318,0.3855,1.0000,0.5442


outcome,gain,loss,unchanged
,0,10,190
,293,4,203


## 6. Check where controls lost evidence

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

All **200 frozen sampled controls** already had all annotated support in embedding top five. This **recorded reference run’s** reranking loses some annotated evidence for **10** questions; **190** preserve recall.

No selected controls have support absent from the top thirty, so that later subgroup is empty. This describes the sampled controls; it does not establish that no such questions exist upstream. Unannotated passages may still support an answer, so generation needs separate evaluation.

In [6]:
view.show('controls')


query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta,outcome
1035915,what is the most delicious steak,control,False,1,1.0000,1.0000,4.0000,1.0000,0.2500,0.4307,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta,outcome
1065453,age of empires definitive edition civilizations,control,False,1,1.0000,1.0000,3.0000,1.0000,0.3333,0.5000,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
1075745,what is a cause of a leg cramp calcium,control,False,1,1.0000,1.0000,2.0000,1.0000,0.5000,0.6309,0.0000,1.0000,10.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
1082927,what does stretching do before a workout,control,False,1,1.0000,1.0000,5.0000,1.0000,0.2000,0.3869,0.0000,1.0000,8.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
1084469,what does a grassroots organization mean,control,False,1,1.0000,1.0000,4.0000,1.0000,0.2500,0.4307,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
1084634,what do trees take in,control,False,1,1.0000,1.0000,5.0000,1.0000,0.2000,0.3869,0.0000,1.0000,7.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
1086528,what association was w.e.b du bois a co-founder of,control,False,1,1.0000,1.0000,5.0000,1.0000,0.2000,0.3869,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
1089845,brewery mineral point,control,False,1,1.0000,1.0000,3.0000,1.0000,0.3333,0.5000,0.0000,1.0000,7.0000,0.0000,0.0000,0.0000,False,-1.0000,loss
49174,batter definition,control,False,1,1.0000,1.0000,5.0000,1.0000,0.2000,0.3869,0.0000,1.0000,6.0000,0.0000,0.0000,0.0000,False,-1.0000,loss


## 7. Define answer-quality evaluation

| Metric | Intuition | Scale |
|---|---|---|
| HF SQuAD token F1 | How much the answer’s words overlap an annotated reference | 0–100% |
| HF SQuAD normalized exact match | Whether the normalized answer matches any reference | 0–100% |
| RAGAS faithfulness | Whether the answer’s claims are supported by its own supplied passages | 0–1 |
| RAGAS answer relevancy | Whether the answer addresses the question | Usually 0–1; cosine can be −1–1 |

Only passage-ID citations are removed for HF SQuAD. Every fixed reference participates in its multiple-reference scoring. Faithfulness uses each path’s own passages. Judge estimates are not human correctness; the judge and generator use the same model, so shared bias is possible.

Truncated answers stay flagged and eligible. Generation failures stay missing. Each metric compares matching successful question pairs and reports its own denominator.

Evaluation attribution: Hugging Face SQuAD and RAGAS.


In [7]:
from shared_scoring.answers import score_squad_answers
from shared_scoring.judge import run_ragas_evaluation
display(Markdown("```python\n" + inspect.getsource(score_squad_answers) + "\n```"))


```python
def score_squad_answers(inputs, metric, citation_pattern, legacy_citation_pattern):
    """Score each saved answer, keeping truncation and the old scorer comparison."""
    records = []
    for row in inputs:
        result = score_squad_answer(metric, row['answer'], row['reference_answers'], row['query_id'], citation_pattern)
        custom = score_custom_answer(row['answer'], row['reference_answers'], legacy_citation_pattern)
        for key, saved in [('token_f1', 'existing_token_f1_0_1'), ('normalized_em', 'existing_normalized_em_0_1')]:
            if saved in row and abs(custom[key] - row[saved]) > 1e-12:
                raise ValueError(f"Existing custom score changed: {row['query_id']} / {row['arm']} / {key}")
        records.append({key: row[key] for key in ['query_id', 'question', 'arm', 'path', 'answer', 'finish_reason', 'truncated',
                                                 'annotated_support_recall_at_5', 'existing_token_f1_0_1', 'existing_normalized_em_0_1']} |
                       {'squad_answer': remove_passage_citations(row['answer'], citation_pattern),
                        'reference_answers': json.dumps(row['reference_answers'], ensure_ascii=False),
                        'squad_exact_match_pct': result['exact_match'], 'squad_token_f1_pct': result['f1'],
                        'token_f1_delta_0_1': result['f1'] / 100 - custom['token_f1'],
                        'normalized_em_delta_0_1': result['exact_match'] / 100 - custom['normalized_em']})
    return pd.DataFrame(records)

```

## 8. Worked examples: generate → HF SQuAD → RAGAS

Evaluate **two worked examples only**: IMEI **1053665** and control **1031584**. Each path generates from its own five passages; both answers receive local SQuAD scoring and RAGAS faithfulness and answer-relevancy judgments. Live execution performs these calls for the current reranking run.

The displayed pairs also appear in Experiment 3; their exact passage IDs, text and order have been checked against this experiment. These two illustrative pairs do not estimate answer quality for all 700 questions.

In [8]:
# Only the two declared worked examples are generated, not the 700-question cohort.
view.show('prompts')
def generate_api(messages, settings, routing=None):
    return client.chat.completions.create(
        model=settings['model'], messages=messages,
        temperature=settings['temperature'], max_tokens=settings['max_tokens'],
        **settings['extras'])

generation_run = tutorial.generate(generate_api)
if generation_run is not None:
    display(generation_run)


**baseline / original**

**reranked / original**

generate: saved for CURRENT LIVE RUN 20261111T032855Z-3fa453244592


{'answers': 4, 'questions': 2, 'run_id': '20261111T032855Z-3fa453244592'}

## 9. Evaluate the worked-example answers

Compare Token F1, faithfulness and answer relevancy for the two example pairs. These are answer-level results, separate from the 700-question retrieval analysis. Experiment 3 evaluates the broader frozen 79-question sample.

In [9]:
score_run = await tutorial.score()
if score_run is not None:
    display(score_run)
view.show('example_scores')
view.show('walkthrough')


.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1031584 baseline answer_relevancy: ok 0.7120765813182596 (15.0s)


1031584 baseline faithfulness: ok 1.0 (16.9s)


1031584 reranked answer_relevancy: ok 0.8406029284111364 (9.5s)


1031584 reranked faithfulness: ok 1.0 (21.3s)


1053665 baseline faithfulness: ok 0.8461538461538461 (19.5s)


... (5 outputs left out)

**1053665: what is imei and where do i find it**

#### baseline

#### reranked

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,baseline,baseline,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, used to identify valid devices and prevent them from being used on a network if stolen. It is typically a 15-digit number, though CDMA networks use a 14-digit MEID, which is derived from the first 14 digits of the IMEI. You can find the IMEI on the device's original packaging, on a sticker under the battery (for phones with removable batteries), on the back of the device (especially for iPhones, iPads, and iPod touches), in the device settings (under General > This Device on iPhones), or in the SIM card slot area on older iPhone models. For Android devices, the IMEI can be found through the device settings or by dialing *#06# on the phone’s keypad [p_f446bafbb6b1eedb4db9], [p_ceb444d4a87718de91ee], [p_16e01af39f1ab9cc5a66], [p_fbac8ec80aec14e774ec], [p_ba06e4d846eb6168282d].",stop,False,0.5000,0.2639,0.0000,"The IMEI (International Mobile Equipment Identity) is a unique identifier assigned to mobile devices, used to identify valid devices and prevent them from being used on a network if stolen. It is typically a 15-digit number, though CDMA networks use a 14-digit MEID, which is derived from the first 14 digits of the IMEI. You can find the IMEI on the device's original packaging, on a sticker under the battery (for phones with removable batteries), on the back of the device (especially for iPhones, iPads, and iPod touches), in the device settings (under General > This Device on iPhones), or in the SIM card slot area on older iPhone models. For Android devices, the IMEI can be found through the device settings or by dialing *#06# on the phone’s keypad , , , , .","[""An International Mobile Equipment Identity number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network, printed on a sticker that is affixed to the phone underneath the battery.""]",0.0000,26.3889,0.0000,0.0000,0.8462,ok,0.8897,ok,opportunity,True,135.0000,Nebius,20261111T032855Z-3fa453244592
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1053665,what is imei and where do i find it,reranked,reranked,"The IMEI (International Mobile Equipment Identity) is a globally unique 15-digit number assigned to all cellular devices that identifies the device within a mobile network [p_71838a0d93f9593a28b4]. You can find the IMEI on the original packaging of your device [p_ceb444d4a87718de91ee], under the battery on many phones [p_f446bafbb6b1eedb4db9], on the back of the device (e.g., original iPhone, iPad cellular model) [p_fbac8ec80aec14e774ec], or in the device settings under General > This Device on iPhones [p_16e01af39f1ab9cc5a66].",stop,False,1.0000,0.4944,0.0000,"The IMEI (International Mobile Equipment Identity) is a globally unique 15-digit number assigned to all cellular devices that identifies the device within a mobile network . You can find the IMEI on the original packaging of your device , under the battery on many phones , on the back of the device (e.g., original iPhone, iPad cellular model) , or in the device settings under General > This Device on iPhones .","[""An International Mobile Equipment Identity number is a globally unique 15 d

### Batch detail

One 700-question batch contains the 500 opportunities and 200 controls. Reranker-stage timing is not end-to-end RAG latency.

In [10]:
view.show('retrieval_summary')
view.show('attempts')


diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta
0.0000,1.0100,1.0000,1.0000,1.8200,1.0000,0.7296,0.7971,0.9500,1.0000,1.9500,0.9500,0.7194,0.7773,0.0000,-0.0500
0.0020,1.1940,0.0638,0.9968,8.2720,0.1280,0.0834,0.0579,0.6080,0.9968,5.2640,0.6420,0.3318,0.3855,1.0000,0.5442


outcome,gain,loss,unchanged
,0,10,190
,293,4,203


stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
reranking,1036720,,1,ok,200,0.5328,0.0000
stage,query_id,arm,attempt,status,http_status,elapsed_seconds,backoff_seconds
reranking,1049603,,1,ok,200,0.3923,0.0000
reranking,1086992,,1,ok,200,0.4948,0.0000
reranking,106620,,1,ok,200,0.4318,0.0000
reranking,1031655,,1,ok,200,0.4362,0.0000
reranking,1086211,,1,ok,200,0.3216,0.0000
reranking,1058271,,1,ok,200,0.4494,0.0000
reranking,1053598,,1,ok,200,0.3139,0.0000
reranking,1084463,,1,ok,200,0.4015,0.0000


### Control example: two answers and their evidence

Inspect **1031584** using each path’s five passages, generated answer and scores. This control has annotated support in the embedding top five; reranking can change the answer without increasing support recall. It is one of the two examples evaluated above.

In [11]:
view.show('walkthrough', query_id='1031584')


**1031584: what is the technology behind spoofing apps**

#### baseline

#### reranked

query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1031584,what is the technology behind spoofing apps,baseline,baseline,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet, allowing for easy manipulation of caller ID information [p_d0692237237e214e45f6]. Additionally, less common methods include emulating the Bell 202 FSK signal (known as orange boxing) and accessing the SS7 network to make calls appear as if they originate from another phone number [p_cff38286a9d88db1a72f]. These technologies enable spoofing by allowing users to falsify the caller ID displayed on a recipient’s phone, though the legality depends on the intent and use [p_7f89a766dddc564217a2].",stop,False,1.0000,0.1446,0.0000,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet, allowing for easy manipulation of caller ID information . Additionally, less common methods include emulating the Bell 202 FSK signal (known as orange boxing) and accessing the SS7 network to make calls appear as if they originate from another phone number . These technologies enable spoofing by allowing users to falsify the caller ID displayed on a recipient’s phone, though the legality depends on the intent and use .","[""The technology behind spoofing apps is call spoofing technologies.""]",0.0000,14.4578,0.0000,0.0000,1.0000,ok,0.7121,ok,control,False,86.0000,Nebius,20261111T032855Z-3fa453244592
query_id,question,arm,path,answer,finish_reason,truncated,annotated_support_recall_at_5,existing_token_f1_0_1,existing_normalized_em_0_1,squad_answer,reference_answers,squad_exact_match_pct,squad_token_f1_pct,token_f1_delta_0_1,normalized_em_delta_0_1,faithfulness,faithfulness_status,answer_relevancy,answer_relevancy_status,subgroup,diagnostic,answer_words,provider,run_id
1031584,what is the technology behind spoofing apps,reranked,reranked,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet, allowing for easy manipulation of Caller ID information [p_d0692237237e214e45f6]. Additionally, spoofing can be achieved through methods like emulating the Bell 202 FSK signal (orange boxing) or accessing the SS7 network to make calls appear to originate from another number [p_cff38286a9d88db1a72f]. In app install fraud, SDK spoofing or replay attacks are used, where malicious actors mimic communication from legitimate app installations [p_949db4d42be13281e4ad]. These technologies enable the disguise of identity in calls or digital interactions, though their misuse can be illegal [p_cff38286a9d88db1a72f].",stop,False,1.0000,0.1250,0.0000,"The technology behind spoofing apps primarily involves VoIP (Voice-over-Internet-Protocol) technology, which enables voice communications over the internet, allowing for easy manipulation of Caller ID information . Additionally, spoofing can be achieved through methods like emulating the Bell 202 FSK signal (orange boxing) or accessing the SS7 network to make calls appear to originate from another number . In app install fraud, SDK spoofing or replay attacks are used, where malicious actors mimic communication from legitimate app installations . These technologies enable the disguise of identity in calls or digital interactions, though their misuse can be illegal .","[""The technology behind spoofing apps is call spoofing technologies.""]",0.0000,12.5000,0.0000,0.0000,1.0000,ok,0.8406,ok,control,False,97.0000,Nebius,20261111T032855Z-3fa453244

### Separate diagnostic: IMEI in the larger corpus

**Recorded reference run:** numerical observations below describe the saved execution. New Live results may differ; the computed tables describe the selected run.

The **10,210-passage baseline corpus has 0/2 annotated supports** for **1053665**. The **152,576-passage corpus** has both supports. In this **recorded reference reranking run**, their embedding → reranked ranks are **1 → 1, 6 → 3**, and **Recall@5 changes 0.5 → 1.0**.

IMEI already belongs to the frozen 500-opportunity + 200-control cohort. This walkthrough adds no observation and does not change the averages.

In [12]:
view.show('contexts')
if MODE != "plan":
    measured = view.retrieval()
    display(measured[measured.query_id.eq("1053665")])
    # Check completion internally without adding an audit/status panel.
    _run_checks = tutorial.finalize()


passage_id,rank,rerank_rank,is_known_support,text,similarity_score,rerank_score
p_f446bafbb6b1eedb4db9,1,1,True,"Find the IMEI/MEID number. The location of this will vary from phone to phone, but the IMEI/MEID number is usually printed on a sticker that is affixed to the phone underneath the battery. If the phone has an IMEI number but you are on a network that uses MEID numbers, ignore the last digit (IMEI is 15 digits, MEID is 14 digits). GSM networks like AT&T and T-Mobile use IMEI numbers. CDMA networks such as Sprint, Verizon, and US Cellular use MEID numbers.",0.7357,0.9023
passage_id,rank,rerank_rank,is_known_support,text,similarity_score,rerank_score
p_ceb444d4a87718de91ee,2,2,False,"If this is the case, and you did not record the IMEI number beforehand, there are still a number of ways to retrieve the number: On your devices packaging The easiest way to find your IMEI number is by returning to your original phone packaging. 1 IMEI numbers are printed on the original packaging of your device. 2 Check the base and sides for your 15 digit IMEI number.",0.7218,0.8628
p_16e01af39f1ab9cc5a66,3,4,False,"IMEI on iPhone and iPad. Enter the phone settings; Then select General In the filed This device you will find the IMEI number. In models iPhone 3G, 3GS, 4 and 4S, you can check the IMEI number on the SIM card socket. In case of CDMA network there is also the MEID number.",0.7061,0.7618
p_fbac8ec80aec14e774ec,4,5,False,"If you have an original iPhone, find the serial number and IMEI engraved on the back. iPad and iPod touch Find the serial number for your iPad Pro, iPad, or iPod touch on the back of the device. On an iPad (cellular model), you'll also find the IMEI. The MEID is the first 14 digits of the IMEI.",0.7045,0.6587
p_ba06e4d846eb6168282d,5,7,False,If you are looking for a few methods that you can use to find the IMEI number for your Android smartphone then look no further. This guide will show you some simple and common ways of locating the IMEI number for pretty much any smartphone running on the Android Operating System.,0.6707,0.1795
p_71838a0d93f9593a28b4,6,3,True,"An IMEI number is a globally unique 15 digit number assigned to all cellular devices, which identifies your device within the mobile network. We can use this number to block a mobile phone from being used by another person or phone company if it has been lost or stolen. Back to Top.",0.6577,0.7653
p_e00aa5b17af86f587618,7,6,False,"The IMEI is only used for identifying the device and has no permanent or semi-permanent relation to the subscriber. Instead, the subscriber is identified by transmission of an International mobile subscriber identity (IMSI) number, which is stored on a SIM card that can in theory be transferred to any handset.",0.6181,0.5764
p_41ac873926ff56ef6e6a,8,9,False,The Sleep/Wake button is on the right side of the device. The device has a solid-state Home button with Touch ID. There's a True Tone LED flash on the back and a SIM tray on the right side that holds a fourth form factor (4FF) nano-SIM card. The IMEI is etched on the SIM tray. See the tech specs for iPhone 7.,0.5629,0.0082
p_8515a786329aa27a5573,9,8,False,"International Mobile Subscriber Identity (IMSI) Definition - What does International Mobile Subscriber Identity (IMSI) mean? An international mobile subscriber identity (IMSI) is a unique number, usually fifteen digits, associated with Global System for Mobile Communications (GSM) and Universal Mobile Telecommunications System (UMTS) network mobile phone users. The IMSI is a unique number identifying a GSM subscriber. This number has two parts.",0.5281,0.0237


,query_id,question,subgroup,diagnostic,gold_count,embedding_recall_at_5,embedding_recall_at_30,embedding_first_support_rank,embedding_hit_at_5,embedding_mrr_at_5,embedding_ndcg_at_5,reranker_recall_at_5,reranker_recall_at_30,reranker_first_support_rank,reranker_hit_at_5,reranker_mrr_at_5,reranker_ndcg_at_5,opportunity,recall_delta,outcome
158,1053665,what is imei and where do i find it,opportunity,True,2,0.5,1.0,1.0,1.0,1.0,0.613147,1.0,1.0,1.0,1.0,1.0,0.919721,True,0.5,gain
